# Classification

**Classification** is a type of supervised learning where the goal is to predict **discrete categories** (labels) instead of continuous values.

---
## Objective: 
By the end of this lab you will be able to:

1. Train and interpret four core classifiers using scikit-learn directly
2. Understand the effect of key hyperparameters on decision boundaries
3. Recognize bias-variance trade-offs from visual evidence
4. Evaluate models using classification reports and confusion matrices



----
## Types of Classification

- **Binary Classification**  
  Predicts **two possible outcomes** (e.g., positive vs negative).  
  - *Example:* Predict if a customer will **churn** or **stay**.

- **Multi-Class Classification**  
  Predicts **three or more mutually exclusive classes**.  
  - Categorize **news articles** into topics (politics, sports, tech)  
  - Classify **images of products** in an e-commerce platform  
  - Predict the **customer segment** based on behavior (low, medium, high value)

## Importance in Data Science

- Helps make **data-driven decisions**.  
- Automates tasks that involve **categorization**.  
- Enables applications like **fraud detection, medical diagnosis, and customer segmentation**.  
- Extracts **actionable insights** by identifying patterns in data.



> **Note on the `Dataset` class**  
> A `Dataset` helper is provided in `utils.py`. It loads a CSV, splits it into train/val/test sets, and reports class imbalance. You do **not** need to modify it 


# Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from utils import Dataset
import mlflow
import warnings
warnings.filterwarnings('ignore')
import utils


# Part 1-  Linear Classification

Linear classification is a type of **supervised learning** where the model predicts categories by combining input features linearly and applying a **decision function**. It is widely used for both **binary** and **multi-class classification**.

---

## 1. Binary Linear Classification

In **binary classification**, the goal is to predict **two classes**.  

A common example is **Logistic Regression**, which uses the **sigmoid function** to output probabilities:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

- Converts linear outputs into probabilities between 0 and 1  
- Thresholding at 0.5 gives class prediction (0 or 1)  

**Binary Cross-Entropy Loss** measures prediction error:

$$
L = -\frac{1}{N} \sum_{i=1}^{N} \big[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \big]
$$

$$
y_i = \text{true label (0 or 1)}, \quad \hat{y}_i = \text{predicted probability}
$$
 

The loss is minimized using **gradient descent** or optimizers like **Adam**, updating the model weights iteratively.

**Sigmoid Function Illustration:**  
![Sigmoid Function Placeholder](photos/sigmoid.png)



---

## 2. Multi-Class Linear Classification

For **multi-class classification** (3 or more classes), the **softmax function** generalizes sigmoid:

$$
\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

- Converts linear outputs into probabilities across all classes  
- Predicted class = the one with **highest probability**  

**Categorical Cross-Entropy Loss** is used:

$$
L = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} y_{i,k} \log(\hat{y}_{i,k})
$$


---

## 3. Linear Classification Workflow

**Step-by-step workflow:**

1. Compute linear combination of inputs:  

$$
z = w^T x + b
$$

2. Apply **activation function**:  
   - Sigmoid → binary  
   - Softmax → multi-class  

3. Compute **loss function**:  
   - Binary cross-entropy (binary)  
   - Categorical cross-entropy (multi-class)  

4. **Optimize weights** using gradient descent or advanced optimizers  



### Binary classification - Churn prediction

In [ ]:
binary_churn_dataset=Dataset(filepath="../data/binary_churn.csv",
                                 target_column="Churn")


# As a sanity check, we print out the size of the training and test data.
print(f"Training data size: {binary_churn_dataset.X_train.shape[0]} samples")
print(f"Test data size: {binary_churn_dataset.X_test.shape[0]} samples")
print(f'Train label shape: {binary_churn_dataset.y_train.shape} ')
print(f'Test label shape: {binary_churn_dataset.y_test.shape} ')



Question 1.1:
Why do we create 3 splits? Can we train on all data without splitting? Why or Why not? 
Your Answer:

Question 1.2:
Suppose a student normalizes the entire dataset before splitting
Explain briefly:
- Is this an issue? Support your claim


In [ ]:
# Visualize the Dataset

plt.figure(figsize=(8, 6))
plt.scatter(
    binary_churn_dataset.X_train["Age"],
    binary_churn_dataset.X_train["Income (in $1000s)"],
    c=binary_churn_dataset.y_train,
    cmap="bwr",
    edgecolor="k",
    alpha=0.7
)
plt.xlabel("Age")
plt.ylabel("Income (in $1000s)")
plt.title("Training Data: Age vs Income (Colored by Churn)")
plt.show()

Train a binary Logistic Regression model

In [ ]:
from sklearn.linear_model import LogisticRegression

# Step 1: Initialize the model with appropriate hyperparameters
logistic_model = LogisticRegression(
    random_state=42,
    max_iter = 200, 
    solver='lbfgs',
    penalty='l2',       # L2 regularization
    C=1.0               # Inverse of regularization strength; smaller values specify stronger regularization
)

# Step 2: Fit the model on the training data
logistic_model.fit(binary_churn_dataset.X_train, binary_churn_dataset.y_train)



Question 1.3:
- What does `.fit` do? What is stored in the model after training? 

Visualize the model

In [ ]:
# Plot the decision boundary
utils.plot_decision_boundary(
    logistic_model,
    binary_churn_dataset.X_train.values,
    binary_churn_dataset.y_train.values,
    feature_names=["Age", "Income (in $1000s)"],
    title="Logistic Regression — Binary (Training Set)"
)

In [ ]:
# Step 3: Validate the model
y_val_pred = utils.evaluate(logistic_model, binary_churn_dataset.X_val.values, binary_churn_dataset.y_val.values)

In [ ]:
# Evaluate on the test set
y_test_pred = logistic_model.predict(binary_churn_dataset.X_test)
print("=== Test Set Performance ===")
print(classification_report(binary_churn_dataset.y_test, y_test_pred))


Question 1.4:

What value of `C` did you choose and why?
What would happen to the decision boundary if you set `c` to 0.001 ? what about `C=1000`?
Your answer:

### Multi class Classification - Customer Segments

In [ ]:
multiclass_dataset = Dataset(
    filepath="../data/customer_segments_churn.csv",
    target_column="Segment",
    random_state=55
)
# Uncomment the code below to visualize the multiclass dataset (colored by segment)
# plt.figure(figsize=(8, 6))
# plt.scatter(
#     multiclass_dataset.X_train["Age"],
#     multiclass_dataset.X_train["Income (in $1000s)"],
#     c=multiclass_dataset.y_train,
#     cmap="Set1",
#     edgecolor="k",
#     alpha=0.7
# )
# plt.xlabel("Age")
# plt.ylabel("Income (in $1000s)")
# plt.title("Training Data: Age vs Income (Colored by Segment)")
# plt.show()

Train a multi class Logisitic regression
Use the same features Age and Incoome and the same hyperpameter choices. 
Sklean handles multi class automatically 

In [ ]:
X_train_m = multiclass_dataset.X_train[["Age", "Income (in $1000s)"]].values
y_train_m = multiclass_dataset.y_train.values
X_val_m   = multiclass_dataset.X_val[["Age", "Income (in $1000s)"]].values
y_val_m   = multiclass_dataset.y_val.values

# TODO: Initialize and train a Logistic regression model for multiclass classification

logistic_multi = None
# Your code here 
utils.plot_decision_boundary(logistic_multi, X_train_m, y_train_m,
                       feature_names=["Age", "Income (in $1000s)"],
                       title="Logistic Regression — Multi-Class (Training Set)",
                       cmap='Set1')

In [ ]:

utils.evaluate(logistic_multi, X_val_m, y_val_m, title="Logistic Multi-Class — Validation")


Question 1.5:
Look at the binary boundary vs. the multi-class boundary. Why is Logistic Regression constrained to produce **linear** decision boundaries? Under what data conditions would this be a fundamental limitation?

# Part 2 - k-Nearest Neighbors (k-NN)

**k-Nearest Neighbors (k-NN)** is a simple **instance-based learning** method.  
It does not have an explicit training phase; instead, it **stores all training data** and makes predictions based on proximity to the stored points.

---

## 1. How k-NN Works

For any test instance:

1. **Find k closest training examples** using a distance metric (e.g., Euclidean distance).  
2. **For classification**: Take a **majority vote** among the k neighbors.  
3. **For probability estimation**: Compute the fraction of neighbors in each class.  

---

## 2. k-NN Variations

To improve efficiency and scalability:

- **Ball Tree / KD Tree**: Efficient spatial indexing for faster neighbor search.  
- **Approximate Nearest Neighbors**: Libraries like **FAISS** or **Annoy** trade some accuracy for speed.  
- **Reduce training set** (Prototype Selection / Condensing): Keep only a subset of **representative training points**.  

### Prototype Selection / Condensing Methods

- **Condensed Nearest Neighbor (CNN)**: Iteratively remove redundant points while keeping the decision boundary intact.  
- **Edited Nearest Neighbor (ENN)**: Remove noisy points that cause misclassification.  

**Benefit**: Smaller dataset → **faster predictions, less memory usage**.  
**Risk**: May lose some accuracy if important points are discarded.

In [ ]:
# Reuse the multi class customer segments dataset
X_train_knn = multiclass_dataset.X_train[["Age", "Income (in $1000s)"]].values
y_train_knn = multiclass_dataset.y_train.values
X_val_knn   = multiclass_dataset.X_val[["Age", "Income (in $1000s)"]].values
y_val_knn   = multiclass_dataset.y_val.values

**TODO  — Train kNN with three different values of k**

Train three separate kNN classifiers with `k = 1`, `k = 5`, and `k = 20`.  
Use `algorithm='ball_tree'` for all three.  
Plot the decision boundary for each and evaluate on the validation set.


In [ ]:
k_values = [1, 5, 20]

knn_models = {}

for k in k_values:
    # TODO: initialize KNeighborsClassifier with n_neighbors=k, algorithm='ball_tree'
    knn = None  # replace with your model

    # TODO: fit the model
    # YOUR CODE HERE

    knn_models[k] = knn

    utils.plot_decision_boundary(knn, X_train_knn, y_train_knn,
                           feature_names=["Age", "Income (in $1000s)"],
                           title=f"kNN Decision Boundary — k={k}",
                           cmap='Set1')

    utils.evaluate(knn, X_val_knn, y_val_knn, title=f"kNN k={k} — Validation")


**Question 2.1 — Bias-Variance trade-off**
Compare the three decision boundaries. Describe what changes visually as k increases.
Which boundary looks overfit? Which looks underfit? How does this relate to bias and variance?

*Your answer here:*
---

**Question 2.2 — Choosing k in practice**
Based on your validation results above, which k would you choose and why?
What is the risk of choosing k based on test set performance instead of validation set performance?

*Your answer here:*


# Part 3 -  Support Vector Machine (SVM)

**Support Vector Machines (SVMs)** 
SVMs find the **maximum-margin hyperplane** — the decision boundary that is as far as possible from the nearest training points of each class. Those nearest points are called **support vectors**.

**Soft-margin SVM** (parameter C):  
Real data is rarely linearly separable. The parameter C controls the trade-off:

- High C → penalizes misclassifications heavily → tight margin, risk of overfitting  
- Low C → tolerates some misclassifications → wider margin, more generalizable

**The Kernel Trick:**  
For non-linearly separable data, SVMs map features into a higher-dimensional space using a **kernel function**, allowing a linear boundary in that space which corresponds to a non-linear boundary in the original space.

| Kernel | When to use |
|--------|-------------|
| `linear` | Data is approximately linearly separable |
| `rbf` | Default choice; works well for most problems |
| `poly` | When data has polynomial structure |

**Kernel SVM Illustration:**  
![Kernel SVM Placeholder](photos/kernel_trick.png)

**RBF kernel gamma (γ):**  
- High γ → each support vector has a **local** influence → wiggly boundary → overfitting risk  
- Low γ → each support vector has a **broad** influence → smoother boundary




In [ ]:
ad_click_dataset = Dataset(
    filepath="../data/ad_click.csv",
    target_column="Clicked_ad"
)
X_train_svm = ad_click_dataset.X_train[["Time_spent", "Pages_visited"]].values
y_train_svm = ad_click_dataset.y_train.values
X_val_svm   = ad_click_dataset.X_val[["Time_spent", "Pages_visited"]].values
y_val_svm   = ad_click_dataset.y_val.values

plt.figure(figsize=(8, 6))
plt.scatter(X_train_svm[:, 0], X_train_svm[:, 1], c=y_train_svm,
            cmap='bwr', edgecolor='k', alpha=0.7)
plt.xlabel("Time_spent")
plt.ylabel("Pages_visited")
plt.title("Ad Click Dataset — Training Points")
plt.show()

TODO: Train an SVM with a linear kernel

Start with a linear kernel to establish a baseline. Choose a value for `C`.


In [ ]:
# TODO: initialize SVC with kernel='linear', choose C, set probability=True
svm_linear = None  # replace with your model

# TODO: fit the model
# YOUR CODE HERE

utils.plot_decision_boundary(svm_linear, X_train_svm, y_train_svm,
                       feature_names=["Time_spent", "Pages_visited"],
                       title="SVM — Linear Kernel")

utils.evaluate(svm_linear, X_val_svm, y_val_svm, title="SVM Linear — Validation")


**TODO Train an SVM with an RBF kernel**

Now try the RBF kernel. Experiment with at least two combinations of `C` and `gamma`.


In [ ]:
# TODO: initialize SVC with kernel='rbf', choose C and gamma, set probability=True
svm_rbf = None  # replace with your model

# TODO: fit the model
# YOUR CODE HERE

utils.plot_decision_boundary(svm_rbf, X_train_svm, y_train_svm,
                       feature_names=["Time_spent", "Pages_visited"],
                       title="SVM — RBF Kernel")

utils.evaluate(svm_rbf, X_val_svm, y_val_svm, title="SVM RBF — Validation")


**Question 3.1 — Linear vs. RBF boundary**
Compare the two decision boundaries. What structural difference do you observe?
Why can the RBF kernel produce this kind of boundary while the linear kernel cannot?

*Your answer here:*


---


# Part 4:  Random Forest 


**Bagging (Bootstrap Aggregating)** reduces model variance by training many models on different random subsamples of the data and aggregating their predictions via majority vote.

**Random Forest** extends bagging by adding **random feature selection** at each split: instead of considering all features, each tree considers only a random subset. This **decorrelates** the trees, making the ensemble more robust.

Key hyperparameters:

| Parameter | Effect |
|-----------|--------|
| `n_estimators` | More trees → more stable predictions, diminishing returns |
| `max_depth` | Deeper trees → more complex, risk of overfitting |
| `max_features` | Fraction of features per split; lower = more decorrelated trees |

**Grid Search** is a systematic way to find good hyperparameters: you define a grid of values and train/evaluate the model at every combination using cross-validation.
**Random Forest Illustration:**  
![Random Forest Placeholder](photos/random-forest.png)


In [ ]:
# Reuse the ad_click dataset
X_train_rf = ad_click_dataset.X_train.values
y_train_rf = ad_click_dataset.y_train.values
X_val_rf   = ad_click_dataset.X_val.values
y_val_rf   = ad_click_dataset.y_val.values


**Question 4.1 — Grid design rationale** *(answer before writing any code)*

- Why would trying `max_depth=100` be a bad idea for a Random Forest?
- Why does adding more estimators beyond a certain point stop helping?

*Your answer here:*


**TODO 4.1 — Run a grid search to find the best Random Forest configuration**

In [ ]:

param_grid = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [3, 5, 7, 10]
}

# TODO: initialize a RandomForestClassifier (base model for grid search)
rf_base = None

# TODO: initialize GridSearchCV with 5-fold CV, scoring='accuracy'
grid_search = None  # replace with your GridSearchCV


# TODO: fit the grid search on training data
# YOUR CODE HERE


print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)



In [ ]:
# Use the best estimator from grid search
best_rf = grid_search.best_estimator_

X_train_rf_2d = ad_click_dataset.X_train[["Time_spent", "Pages_visited"]].values
y_train_rf_2d = ad_click_dataset.y_train.values

# Refit on 2D for visualization only
best_rf_2d = RandomForestClassifier(**grid_search.best_params_, random_state=42)
best_rf_2d.fit(X_train_rf_2d, y_train_rf_2d)

utils.plot_decision_boundary(best_rf_2d, X_train_rf_2d, y_train_rf_2d,
                       feature_names=["Time_spent", "Pages_visited"],
                       title=f"Random Forest — Best Params: {grid_search.best_params_}",
                       cmap='bwr')

# Evaluate the full model (all features) on validation set
utils.evaluate(best_rf, X_val_rf, y_val_rf, title="Random Forest — Validation")



**TODO — Feature importance**

Random Forest provides built-in feature importance scores. Extract and plot them.


In [ ]:
feature_names_rf = ad_click_dataset.X_train.columns.tolist()

# TODO: extract feature importances from best_rf and plot a bar chart
# Hint - use best_rf.feature_importances_ to get importance scores

importances =None
# Your code here


# References

- [scikit-learn](https://scikit-learn.org/) – Machine learning library for Python  
- [XGBoost](https://xgboost.readthedocs.io/) – Gradient boosting library for structured/tabular data  
- [Matplotlib](https://matplotlib.org/) – Plotting and visualization library  
- [Pandas](https://pandas.pydata.org/) – Data manipulation and analysis library  
- [NumPy](https://numpy.org/) – Numerical computing library  
- [Imbalanced-learn](https://imbalanced-learn.org/) – Library for handling class imbalance in datasets
